# Q4: Logistic Regression from Scratch - BSDS500 Boundary Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat

BASE = r"C:\Users\cqds\Downloads\bsds500archive"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
IMAGES_TRAIN = os.path.join(BASE, "images", "train")
IMAGES_TEST = os.path.join(BASE, "images", "test")
GT_TRAIN = os.path.join(BASE, "ground_truth", "train")
GT_TEST = os.path.join(BASE, "ground_truth", "test")

### Helper functions for loading images and ground truth

In [ ]:
def list_image_files(images_dir, max_images):
    filenames = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    return filenames[:max_images]

def find_gt_path(gt_dir, image_filename):
    stem = os.path.splitext(image_filename)[0]
    candidate = os.path.join(gt_dir, stem + ".mat")
    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"No matching ground-truth .mat file found for image '{image_filename}' "
            f"(expected {candidate})"
        )
    return candidate

def load_bsds_ground_truth(mat_path):
    mat = loadmat(mat_path)
    gt_struct = mat["groundTruth"]
    n_annotators = gt_struct.shape[1]
    boundary_maps = [np.asarray(gt_struct[0, i]["Boundaries"][0, 0], dtype=float)
                      for i in range(n_annotators)]
    consensus = np.mean(boundary_maps, axis=0)
    return (consensus >= 0.5).astype(int)

def get_boundary_labels(gt_array):
    unique_vals = np.unique(gt_array)
    if len(unique_vals) <= 2:
        return (gt_array > 0).astype(int)
    diff_right = gt_array[:, :-1] != gt_array[:, 1:]
    diff_down = gt_array[:-1, :] != gt_array[1:, :]
    boundary = np.zeros_like(gt_array, dtype=bool)
    boundary[:, :-1] |= diff_right
    boundary[:-1, :] |= diff_down
    return boundary.astype(int)

def extract_pixel_features(img_array):
    gray = img_array.mean(axis=2)
    grad_y = np.abs(np.diff(gray, axis=0, prepend=gray[:1, :]))
    grad_x = np.abs(np.diff(gray, axis=1, prepend=gray[:, :1]))
    grad_mag = np.sqrt(grad_x ** 2 + grad_y ** 2)
    feats = np.stack([img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2], gray, grad_mag], axis=-1)
    return feats.reshape(-1, 5)

def load_pixel_dataset(images_dir, gt_dir, max_images, pixels_per_image, seed=0):
    rng = np.random.RandomState(seed)
    filenames = list_image_files(images_dir, max_images)
    X_parts, y_parts = [], []
    total_pixels = 0
    for fname in filenames:
        img = np.array(Image.open(os.path.join(images_dir, fname)).convert("RGB"), dtype=float) / 255.0
        gt_path = find_gt_path(gt_dir, fname)
        gt = load_bsds_ground_truth(gt_path)
        if gt.shape != img.shape[:2]:
            raise ValueError(
                f"Ground truth shape {gt.shape} does not match image shape "
                f"{img.shape[:2]} for '{fname}'"
            )
        labels_full = get_boundary_labels(gt).reshape(-1)
        feats_full = extract_pixel_features(img)
        total_pixels += labels_full.shape[0]
        boundary_idx = np.where(labels_full == 1)[0]
        nonboundary_idx = np.where(labels_full == 0)[0]
        rng.shuffle(boundary_idx)
        rng.shuffle(nonboundary_idx)
        idx = np.concatenate([boundary_idx[:pixels_per_image], nonboundary_idx[:pixels_per_image]])
        X_parts.append(feats_full[idx])
        y_parts.append(labels_full[idx])
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    return X, y, total_pixels, len(filenames)

feature_names = ["R", "G", "B", "gray", "gradient_magnitude"]

### Load training and test pixel samples

In [ ]:
X_train, y_train, total_train_pixels, n_train_images = load_pixel_dataset(
    IMAGES_TRAIN, GT_TRAIN, max_images=40, pixels_per_image=250, seed=0)
X_test, y_test, total_test_pixels, n_test_images = load_pixel_dataset(
    IMAGES_TEST, GT_TEST, max_images=15, pixels_per_image=150, seed=1)

print("Training images used:", n_train_images, " total pixels:", total_train_pixels)
print("Training sample shape:", X_train.shape, " Test sample shape:", X_test.shape)

### Standardize features

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

### Sigmoid

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("sigmoid(0) =", sigmoid(0))

### Gradient descent training

In [ ]:
n_samples, n_features = X_train_scaled.shape
weights = np.zeros(n_features)
bias = 0.0
learning_rate = 0.5
n_iterations = 1500
cost_history = []

for i in range(n_iterations):
    z = X_train_scaled.dot(weights) + bias
    p = sigmoid(z)
    cost = -np.mean(y_train * np.log(p + 1e-9) + (1 - y_train) * np.log(1 - p + 1e-9))
    cost_history.append(cost)
    error = p - y_train
    dw = X_train_scaled.T.dot(error) / n_samples
    db = np.sum(error) / n_samples
    weights -= learning_rate * dw
    bias -= learning_rate * db
    if (i + 1) % 300 == 0:
        print("iteration", i + 1, "cost =", round(cost, 4))

print("Final cost:", round(cost_history[-1], 4))

### Test predictions

In [ ]:
p_test = sigmoid(X_test_scaled.dot(weights) + bias)
y_pred = (p_test >= 0.5).astype(int)
accuracy = np.mean(y_pred == y_test)
tp = np.sum((y_test == 1) & (y_pred == 1))
tn = np.sum((y_test == 0) & (y_pred == 0))
fp = np.sum((y_test == 0) & (y_pred == 1))
fn = np.sum((y_test == 1) & (y_pred == 0))
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print("Test accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4), " Recall:", round(recall, 4))

### Feature weights

In [ ]:
weight_table = pd.DataFrame({"feature": feature_names, "weight": weights})
print(weight_table.sort_values("weight", key=abs, ascending=False))

### Plot cost curve

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(cost_history)
plt.xlabel("Iteration")
plt.ylabel("Cost (binary cross-entropy)")
plt.title("Logistic Regression - Gradient Descent Convergence\nImage boundary detection")
plt.grid(alpha=0.3)
plt.show()